# 2D DEXSY Colab Demo

This notebook loads the bundled 2D inverse model checkpoint, generates a paper-style synthetic forward-model sample, and runs inference without retraining.

In [ ]:
from pathlib import Path
import sys
import csv

import matplotlib.pyplot as plt
import numpy as np
import torch


def find_repo_root(start=None):
    start_path = Path.cwd() if start is None else Path(start)
    candidates = [start_path] + list(start_path.resolve().parents)
    for candidate in candidates:
        if (candidate / 'improved_2d_dexsy').exists() and (candidate / 'checkpoints').exists():
            return candidate
    raise FileNotFoundError('Could not find the repo root. Run `%cd /content/FYP` first.')


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from improved_2d_dexsy import ForwardModel2D, predict_from_signal, resolve_checkpoint_path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Repo root: {ROOT}')
print(f'Device: {device}')
print(f'Default checkpoint: {resolve_checkpoint_path()}')


In [ ]:
forward_model = ForwardModel2D(n_d=64, n_b=64)
checkpoint_path = resolve_checkpoint_path()
print(f'Forward model grid: {forward_model.n_b}x{forward_model.n_b} signal -> {forward_model.n_d}x{forward_model.n_d} spectrum')


In [ ]:
true_distribution, noisy_signal, params, clean_signal = forward_model.generate_sample(
    n_compartments=2,
    return_reference_signal=True,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(true_distribution, cmap='hot', origin='lower')
axes[0].set_title('True 2D Distribution')
axes[0].set_xlabel('D2 index')
axes[0].set_ylabel('D1 index')

axes[1].imshow(noisy_signal, cmap='viridis', origin='lower')
axes[1].set_title('Noisy Forward Signal')
axes[1].set_xlabel('b2 index')
axes[1].set_ylabel('b1 index')
plt.tight_layout()

print('Sample parameters:')
print(f"  mixing_time = {params['mixing_time']:.4f} s")
print(f"  noise_sigma = {params['noise_sigma']:.4f}")
print(f"  baseline_snr = {params['baseline_snr']:.2f}")


In [ ]:
result = predict_from_signal(
    noisy_signal,
    checkpoint_path=checkpoint_path,
    device=device,
    forward_model=forward_model,
    true_spectrum=true_distribution,
)

display(result.figure)
print('Summary metrics:')
for key, value in result.summary_metrics.items():
    if isinstance(value, float):
        print(f'  {key}: {value:.6f}')
    else:
        print(f'  {key}: {value}')


In [ ]:
log_path = ROOT / 'checkpoints' / 'training_log.csv'
epochs = []
train_loss = []
val_loss = []

with log_path.open() as handle:
    reader = csv.DictReader(handle)
    for row in reader:
        epochs.append(int(row['epoch']))
        train_loss.append(float(row['train_loss']))
        val_loss.append(float(row['val_loss']))

plt.figure(figsize=(8, 5))
plt.plot(epochs, train_loss, label='Train Loss', linewidth=2)
plt.plot(epochs, val_loss, label='Val Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Bundled Training Curve')
plt.grid(alpha=0.3)
plt.legend()
plt.show()